# Porownanie architektur pod domain shift (RarePlanes) — Colab

Trening na **podzbiorze 10k synthetic** (stratyfikowany po 15 lotniskach, ta sama lista plikow co lokalnie), ewaluacja cross-domain na **realnym holdoucie**.

Architektury: YOLOv10n (referencja, wynik lokalny: mAP@50 0.459 na 10k) vs RT-DETR-l vs (opcjonalnie D-FINE).

**Wymagania:** GPU runtime (najlepiej A100). Dane ~31 GB (30 GB synthetic 10k + 0.9 GB real test) — mieszcza sie na dysku Colaba.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 1. Repo + zaleznosci
!git clone https://github.com/JakubPaszke/rareplanes-domain-shift.git
%cd rareplanes-domain-shift
!pip -q install ultralytics pycocotools

In [ ]:
# 2. Dane: adnotacje + realny test (tarball) + synthetic 10k wg listy plikow
import os
os.makedirs('data/real/annotations', exist_ok=True)
os.makedirs('data/real/tarballs', exist_ok=True)
os.makedirs('data/synthetic/annotations', exist_ok=True)
os.makedirs('data/synthetic/images/train', exist_ok=True)

B='https://rareplanes-public.s3.amazonaws.com'
# adnotacje (male)
for f in ['instances_train_aircraft.json','instances_test_aircraft.json']:
    !curl -sf -o data/real/annotations/{f} {B}/real/metadata_annotations/{f}
    !curl -sf -o data/synthetic/annotations/{f} {B}/synthetic/metadata_annotations/{f}
# realny test (tiled tarball 889 MB)
!curl -sf -o data/real/tarballs/test.tar.gz {B}/real/tarballs/test/RarePlanes_test_PS-RGB_tiled.tar.gz
!mkdir -p data/real/PS-RGB_tiled && tar -xzf data/real/tarballs/test.tar.gz -C data/real/PS-RGB_tiled/
!ls data/real/PS-RGB_tiled/PS-RGB_tiled | head -3

In [ ]:
# 3. Synthetic 10k: rownolegle pobieranie wg list (configs/synthetic_10k_*.txt)
#    ~30 GB; na Colabie zwykle 10-30 MB/s -> 20-50 min
!cat configs/synthetic_10k_train_files.txt configs/synthetic_10k_val_files.txt > /tmp/files.txt
!wc -l /tmp/files.txt
!cat /tmp/files.txt | xargs -P16 -I{} sh -c 'curl -sf -o "data/synthetic/images/train/{}.part" "'{B}'/synthetic/train/images/{}" && mv "data/synthetic/images/train/{}.part" "data/synthetic/images/train/{}" || rm -f "data/synthetic/images/train/{}.part"'
!ls data/synthetic/images/train | wc -l

In [ ]:
# 4. Konwersja COCO->YOLO (pominie obrazy, ktorych nie pobralismy)
!python3 src/coco_to_yolo.py --domain synthetic --classes aircraft --val-frac 0.15

In [ ]:
# 5a. RT-DETR-l — trening (natywny Linux: workers=8 OK)
!python3 src/train_yolo.py --data data/yolo/synthetic_aircraft/data.yaml \
  --name rtdetr_l_10k --model rtdetr-l.pt \
  --epochs 60 --batch 16 --imgsz 512 --seed 42 --patience 20 --workers 8 \
  --val-data data/yolo/synthetic_aircraft/data.yaml

In [ ]:
# 5b. Ewaluacja cross-domain na realnym tescie (AP per-size)
!python3 src/eval_per_size.py --weights runs/rtdetr_l_10k/weights/best.pt \
  --img-dir data/real/PS-RGB_tiled/PS-RGB_tiled \
  --coco-gt data/real/annotations/instances_test_aircraft.json --name rtdetr_l_10k_ml

## Wyniki
Skopiuj `results/per_size/*.json` do repo (commit ze swojego konta) albo pobierz lokalnie.

Punkty odniesienia (10k, YOLOv10n, lokalnie): mAP@50 **0.459** (slaby HSV) / 0.452 (baseline 45k).

TODO (do uzgodnienia): D-FINE; pomiar FPS na tym samym sprzecie dla uczciwego rankingu koszt-jakosc.